# <center> 各大模型网关平台langchain接入指南

In [ ]:
from langchain_openai import ChatOpenAI 
from langchain_core.messages import AIMessage, AIMessageChunk
from langchain_core.language_models.chat_models import ChatResult



## 1.1 DeepSeek-Chat模型的接入

In [ ]:
class DeepSeekChatModel(ChatOpenAI):
    def __init__(
        self, 
        model: str, 
        api_key: str, 
        base_url: str, 
        temperature: float = 0.7,  # 增加默认值，和父类一致
        **kwargs  # 兼容父类的其他参数（如 max_tokens、timeout 等）
    ):
        # 调用父类初始化，传递所有参数
        super().__init__(
            model=model, 
            api_key=api_key, 
            base_url=base_url, 
            temperature=temperature,
            **kwargs  # 传递其他参数，保证兼容性
        )

## 1.2 DeepSeek-Reasoner模型的接入

In [ ]:
class DeepSeekReasoningModel(ChatOpenAI):
    
    def _get_request_payload(
        self, 
        input_, 
        *, 
        stop=None, 
        **kwargs
    ):
        """Ensure assistant messages include reasoning_content for DeepSeek thinking + tool_calls API."""
        payload = super()._get_request_payload(input_, stop=stop, **kwargs)
        messages = self._convert_input(input_).to_messages()
        for i, m in enumerate(messages):
            if i < len(payload.get("messages", [])) and isinstance(m, AIMessage):
                # DeepSeek requires reasoning_content on assistant messages in thinking mode
                payload["messages"][i]["reasoning_content"] = (
                    (m.additional_kwargs or {}).get("reasoning_content") or ""
                )
        return payload

    def _create_chat_result(
        self,
        response,
        generation_info: dict | None = None,
    ) -> ChatResult:
        """Process non-streaming response, extract reasoning_content"""
        result = super()._create_chat_result(response, generation_info)
        if not result.generations:
            return result
        if hasattr(response, "choices") and response.choices:
            msg = response.choices[0].message
            if hasattr(msg, "reasoning_content") and msg.reasoning_content:
                result.generations[0].message.additional_kwargs["reasoning_content"] = (
                    msg.reasoning_content
                )
        return result

    def _convert_chunk_to_generation_chunk(
        self, chunk, default_chunk_class, base_generation_info
    ):
        """Stream chunk to generation chunk"""
        gen = super()._convert_chunk_to_generation_chunk(
            chunk, default_chunk_class, base_generation_info
        )
        if gen is None:
            return None
        choices = chunk.get("choices", []) or chunk.get("chunk", {}).get("choices", [])
        if choices:
            delta = choices[0].get("delta") or {}
            if delta.get("reasoning_content"):
                msg = gen.message
                if isinstance(msg, AIMessageChunk):
                    msg.additional_kwargs["reasoning_content"] = delta["reasoning_content"]
        return gen


(1) _create_chat_result：非流式提取 reasoning_content

```python
    def _create_chat_result(
        self,
        response,
        generation_info: dict | None = None,
    ) -> ChatResult:
        result = super()._create_chat_result(response, generation_info)
        if not result.generations:
            return result
        if hasattr(response, "choices") and response.choices:
            msg = response.choices[0].message
            if hasattr(msg, "reasoning_content") and msg.reasoning_content:
                result.generations[0].message.additional_kwargs["reasoning_content"] = (
                    msg.reasoning_content
                )
        return result
```
这是“响应解析阶段”的兼容扩展字段保存。

- 标准 `OpenAI.chat.completions 的 assistant message 通常只有 content，而 DeepSeek 思考模式会在同一层级多一个 reasoning_content。（DeepSeek官方文档 https://api-docs.deepseek.com/guides/thinking_mode#tool-calls 是这么定义的：和 content 同级）。

```json
{
    "role": "assistant", 
    "content": "...", 
    "reasoning_content": "..."
}

```

- LangChain 的 ChatOpenAI 在解析响应时，会把 choices[0].message 变成 AIMessage，但不会自动把供应商扩展字段保留下来。
这里做的事就是：从 provider 原始响应对象中把 reasoning_content 抠出来，放进 LangChain 的 AIMessage.additional_kwargs，方便后续在流式输出时读取、展示“思考 token”。

（2）_convert_chunk_to_generation_chunk：流式从 delta 提取 reasoning_content

```python
    def _convert_chunk_to_generation_chunk(
        self, chunk, default_chunk_class, base_generation_info
    ):
        gen = super()._convert_chunk_to_generation_chunk(
            chunk, default_chunk_class, base_generation_info
        )
        if gen is None:
            return None
        choices = chunk.get("choices", []) or chunk.get("chunk", {}).get("choices", [])
        if choices:
            delta = choices[0].get("delta") or {}
            if delta.get("reasoning_content"):
                msg = gen.message
                if isinstance(msg, AIMessageChunk):
                    msg.additional_kwargs["reasoning_content"] = delta["reasoning_content"]
        return gen
```

- 在 流式接口里（SSE），模型是“分块”吐出 delta 的；DeepSeek 会把思考过程也分块放在 delta.reasoning_content 里。
- LangChain 会把每个 chunk 转成 AIMessageChunk（部分内容），你在这里把 delta["reasoning_content"] 挂到 AIMessageChunk.additional_kwargs，这样你在上层 streaming loop 就能像读普通 token 一样读 reasoning token。

这相当于 OpenAI streaming 里读 delta.content，只不过 DeepSeek 还多了 delta.reasoning_content 这一条通道。

(3) _get_request_payload：关键点——把 reasoning_content 回传给 API

```python
def _get_request_payload(...):
    payload = super()._get_request_payload(input_, stop=stop, **kwargs)
    messages = self._convert_input(input_).to_messages()
    for i, m in enumerate(messages):
        if i < len(payload.get("messages", [])) and isinstance(m, AIMessage):
            payload["messages"][i]["reasoning_content"] = (
                (m.additional_kwargs or {}).get("reasoning_content") or ""
            )
    return payload
```

### 为什么要做这一步（对照“标准 OpenAI SDK”）
- OpenAI 官方 Python SDK（chat.completions）构造 messages 时，只认识标准字段：role/content，以及工具相关 tool_calls / tool 消息等。
- LangChain 的 ChatOpenAI 在发请求时，会把 AIMessage 转成 dict（通常只保留 OpenAI 规范字段，如 role/content/tool_calls/...），不会自动把 additional_kwargs 里的自定义字段（比如 reasoning_content）放进请求体。
- 但 DeepSeek 思考模式 + tool calls 有一个兼容性要求：当模型在一次回答过程中触发 tool_calls，在后续子回合把上一条 assistant（包含 tool_calls）发回去继续推理时，那条 assistant 必须带 reasoning_content 字段，否则它会 400（DeepSeek 文档要求）。
所以 _get_request_payload 的作用是：在“请求序列化阶段”，强行把 AIMessage.additional_kwargs["reasoning_content"] 注入到 payload["messages"][i]，确保 DeepSeek 能继续思考链路。